In [2]:
#Part 1| 1.1
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from collections import Counter
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

from datasets import load_dataset

dataset = load_dataset("glue", "sst2")

print(f"Train size : {len(dataset['train'])}")
print(f"Val size : {len(dataset['validation'])}")
print(f"\nSample:")
print(dataset["train"][0])


Using device: cpu


c:\Users\prosi\Documents\GitHub\NLP_Course\envire\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train size : 67349
Val size : 872

Sample:
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}


In [3]:
#1.2
import pandas as pd

train_df = pd.DataFrame(dataset["train"])

# Class distribution
print("Label counts:")
print(train_df["label"].value_counts())
print(f"\nPositive ratio: {train_df['label'].mean():.2%}")

# Sentence lengths (in words)
train_df["num_words"] = train_df["sentence"].apply(lambda s:
len(s.split()))
print(f"\nSentence length stats:")
print(train_df["num_words"].describe())

# A few examples
print("\n--- Positive examples ---")
for s in train_df[train_df["label"] == 1]["sentence"].sample(3,
random_state=42).values:
 print(f" {s}")
 
print("\n--- Negative examples ---")
for s in train_df[train_df["label"] == 0]["sentence"].sample(3,
random_state=42).values:
 print(f" {s}")


Label counts:
label
1    37569
0    29780
Name: count, dtype: int64

Positive ratio: 55.78%

Sentence length stats:
count    67349.000000
mean         9.409553
std          8.073806
min          1.000000
25%          3.000000
50%          7.000000
75%         13.000000
max         52.000000
Name: num_words, dtype: float64

--- Positive examples ---
 acted meditation on both the profoundly devastating events of one year ago and the slow , painful healing process that has followed in their wake 
 this odd , poetic road movie , spiked by jolts of pop music , pretty much takes place in morton 's ever-watchful gaze -- and it 's a tribute to the actress , and to her inventive director , that the journey is such a mesmerizing one . 
 directed with purpose and finesse by england 's roger mitchell , who handily makes the move from pleasing , relatively lightweight commercial fare such as notting hill to commercial fare with real thematic heft . 

--- Negative examples ---
 a dull , dumb and der

In [4]:
#Part2 | 2.1
def tokenize(sentence):
 """Simple lowercase whitespace tokenizer."""
 return sentence.lower().split()
# Verify
print(tokenize("A stirring , funny and finally transporting re-imagining."))



['a', 'stirring', ',', 'funny', 'and', 'finally', 'transporting', 're-imagining.']


In [5]:
#2.2
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
PAD_IDX = 0
UNK_IDX = 1

def build_vocab(dataset_split, min_freq=2):
 """
 Build word-to-index mapping from training data.
 Args:
 dataset_split: HuggingFace dataset split (e.g., dataset["train"])
 min_freq: minimum word frequency to include in vocab
 Returns:
 word2idx: dict mapping word -> index
 idx2word: dict mapping index -> word
 """
 counter = Counter()
 for example in dataset_split:
    tokens = tokenize(example["sentence"])
    counter.update(tokens)
 # TODO: Build word2idx with special tokens at indices 0 and 1
 # Hint: start with {PAD_TOKEN: 0, UNK_TOKEN: 1}
 # then add words with count >= min_freq
 word2idx = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
 idx = 2
 for word, count in counter.items():
        if count >= min_freq:
            word2idx[word] = idx
            idx += 1
 
 idx2word = {i: w for w, i in word2idx.items()}
 return word2idx, idx2word

word2idx, idx2word = build_vocab(dataset["train"], min_freq=2)
V = len(word2idx)
print(f"Vocabulary size: {V}")
print(f"Sample: {dict(list(word2idx.items())[2:7])}")


Vocabulary size: 14311
Sample: {'hide': 2, 'new': 3, 'secretions': 4, 'from': 5, 'the': 6}


In [6]:
#2.3
MAX_LEN = 50 # truncate / pad all sentences to this length
def encode_sentence(sentence, word2idx, max_len=MAX_LEN):
 """Convert sentence string to padded list of token IDs."""
 tokens = tokenize(sentence)
 ids = [word2idx.get(t, UNK_IDX) for t in tokens]

 # Truncate
 ids = ids[:max_len]
 # Pad
 pad_length = max_len - len(ids)
 ids = ids + [PAD_IDX] * pad_length

 return ids

class SST2Dataset(Dataset):
 """PyTorch Dataset wrapper for SST-2."""
 def __init__(self, hf_split, word2idx, max_len=MAX_LEN):
    self.data = hf_split
    self.word2idx = word2idx
    self.max_len = max_len
 
 def __len__(self):
    return len(self.data)
 
 def __getitem__(self, idx):
    example = self.data[idx]
    input_ids = encode_sentence(example["sentence"], self.word2idx, self.max_len)
    label = example["label"]
    return torch.tensor(input_ids, dtype=torch.long),torch.tensor(label, dtype=torch.long)

# Create datasets and dataloaders
BATCH_SIZE = 64

train_dataset = SST2Dataset(dataset["train"], word2idx)
val_dataset = SST2Dataset(dataset["validation"], word2idx)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,shuffle=False)

# Sanity check
batch_x, batch_y = next(iter(train_loader))
print(f"Batch input shape : {batch_x.shape}") # [64, 50]
print(f"Batch label shape : {batch_y.shape}") # [64]
print(f"First input IDs : {batch_x[0][:15]}")
print(f"First label : {batch_y[0].item()}")


Batch input shape : torch.Size([64, 50])
Batch label shape : torch.Size([64])
First input IDs : tensor([4087,  470,   83,    6,  975,   20, 4088,  470, 4089,   92,   18, 4090,
          71,    0,    0])
First label : 1


In [7]:
# Part 3 | 3.1
class BiLSTMSentiment(nn.Module):
    """
    Bidirectional LSTM for binary sentiment classification.

    Architecture:
    Embedding -> BiLSTM -> Dropout -> FC -> logits

    We use the concatenation of the final forward and backward
    hidden states as the sentence representation.
    """

    def __init__(self, vocab_size, emb_dim, hidden_dim, num_layers,
                 num_classes=2, dropout=0.3, pad_idx=PAD_IDX):
        
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            emb_dim,
            padding_idx=pad_idx
        )

        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.dropout = nn.Dropout(dropout)

        # BiLSTM produces hidden_dim * 2 (forward + backward)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        """
        Args:
            x: input token IDs, shape [batch_size, seq_len]

        Returns:
            logits: shape [batch_size, num_classes]
        """

        # x: [batch, seq_len]
        embedded = self.dropout(self.embedding(x))  # [batch, seq_len, emb_dim]

        output, (hidden, cell) = self.lstm(embedded)

        # hidden: [num_layers * 2, batch, hidden_dim]

        # Grab the last layer's forward (index -2) and backward (index -1)
        h_forward = hidden[-2, :, :]   # [batch, hidden_dim]
        h_backward = hidden[-1, :, :]  # [batch, hidden_dim]

        h_cat = torch.cat([h_forward, h_backward], dim=1)  # [batch, hidden_dim * 2]

        logits = self.fc(self.dropout(h_cat))  # [batch, num_classes]

        return logits

In [8]:
#3.2
EMB_DIM = 100
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.3
NUM_CLASSES = 2

model = BiLSTMSentiment(
 vocab_size=V,
 emb_dim=EMB_DIM,
 hidden_dim=HIDDEN_DIM,
 num_layers=NUM_LAYERS,
 num_classes=NUM_CLASSES,
 dropout=DROPOUT,
 pad_idx=PAD_IDX,
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if
p.requires_grad)
print(f"Total parameters : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"\nModel:\n{model}")


Total parameters : 2,062,398
Trainable parameters : 2,062,398

Model:
BiLSTMSentiment(
  (embedding): Embedding(14311, 100, padding_idx=0)
  (lstm): LSTM(100, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=2, bias=True)
)


In [9]:
#3.3
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [10]:
#Part 4 | 4.1
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    """Train for one epoch, return average loss and accuracy."""

    model.train()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch_x, batch_y in dataloader:

        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()

        logits = model(batch_x)

        loss = criterion(logits, batch_y)

        loss.backward()

        # Gradient clipping to prevent exploding gradients
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)

        preds = logits.argmax(dim=1)

        total_correct += (preds == batch_y).sum().item()

        total_examples += batch_x.size(0)

    avg_loss = total_loss / total_examples
    accuracy = total_correct / total_examples

    return avg_loss, accuracy


def evaluate(model, dataloader, criterion, device):
    """Evaluate model, return average loss and accuracy."""

    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    with torch.no_grad():

        for batch_x, batch_y in dataloader:

            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            logits = model(batch_x)

            loss = criterion(logits, batch_y)

            total_loss += loss.item() * batch_x.size(0)

            preds = logits.argmax(dim=1)

            total_correct += (preds == batch_y).sum().item()

            total_examples += batch_x.size(0)

    avg_loss = total_loss / total_examples
    accuracy = total_correct / total_examples

    return avg_loss, accuracy

In [11]:
#4.2
NUM_EPOCHS = 5

print(f"{'Epoch':>5} {'Train Loss':>10} {'Train Acc':>9} {'Val Loss':>10} {'Val Acc':>9}")
print("-" * 55)

for epoch in range(1, NUM_EPOCHS + 1):
 train_loss, train_acc = train_one_epoch(model, train_loader,optimizer, criterion, device)
 val_loss, val_acc = evaluate(model, val_loader, criterion, device)
 print(f"{epoch:5d} {train_loss:10.4f} {train_acc:8.2%}{val_loss:10.4f} {val_acc:8.2%}")

Epoch Train Loss Train Acc   Val Loss   Val Acc
-------------------------------------------------------
    1     0.5526   70.67%    0.4653   78.10%
    2     0.3661   83.52%    0.5137   79.36%
    3     0.2817   88.16%    0.4705   81.54%
    4     0.2335   90.52%    0.5341   81.88%
    5     0.1997   91.94%    0.4907   81.08%


In [12]:
#Part 5 | 5.1
def predict_sentiment(model, sentence, word2idx, device):
    """
    Predict sentiment for a single sentence.

    Returns:
        label: "Positive" or "Negative"
        confidence: probability of predicted class
    """

    model.eval()

    input_ids = encode_sentence(sentence, word2idx)

    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

    with torch.no_grad():

        logits = model(input_tensor)

        probs = torch.softmax(logits, dim=1)

        pred = logits.argmax(dim=1).item()

        conf = probs[0, pred].item()

    label_str = "Positive" if pred == 1 else "Negative"

    return label_str, conf


# Test sentences — try your own!
test_sentences = [
    "This movie was absolutely wonderful and deeply moving .",
    "A total waste of time , boring and predictable .",
    "Not great , not terrible , just okay .",
    "Although the plot was weak , the acting was superb .",
    "I hated every minute of this dreadful film .",
    "An unexpectedly delightful surprise .",
]

print(f"{'Prediction':>10} {'Conf':>6} Sentence")
print("-" * 70)

for sent in test_sentences:

    label, conf = predict_sentiment(model, sent, word2idx, device)

    print(f"{label:>10} {conf:5.1%} {sent}")

Prediction   Conf Sentence
----------------------------------------------------------------------
  Positive 100.0% This movie was absolutely wonderful and deeply moving .
  Negative 99.7% A total waste of time , boring and predictable .
  Negative 65.7% Not great , not terrible , just okay .
  Negative 98.8% Although the plot was weak , the acting was superb .
  Negative 94.6% I hated every minute of this dreadful film .
  Positive 100.0% An unexpectedly delightful surprise .


In [13]:
#5.2
def get_errors(model, hf_split, word2idx, device, max_errors=10):
    """Find misclassified examples from a dataset split."""

    model.eval()

    errors = []

    for example in hf_split:

        sentence = example["sentence"]
        true_label = example["label"]

        pred_label, conf = predict_sentiment(model, sentence, word2idx, device)

        pred_int = 1 if pred_label == "Positive" else 0

        if pred_int != true_label:

            errors.append({
                "sentence": sentence,
                "true": "Pos" if true_label == 1 else "Neg",
                "pred": pred_label,
                "conf": conf,
            })

        if len(errors) >= max_errors:
            break

    return errors


errors = get_errors(model, dataset["validation"], word2idx, device, max_errors=10)

print(f"\n--- First {len(errors)} misclassified validation examples ---\n")

for e in errors:
    print(f" True: {e['true']:>3} | Pred: {e['pred']:>8} ({e['conf']:.1%}) | {e['sentence']}")


--- First 10 misclassified validation examples ---

 True: Neg | Pred: Positive (71.8%) | even horror fans will most likely not find what they 're seeking with trouble every day ; the movie lacks both thrills and humor . 
 True: Neg | Pred: Positive (53.2%) | pumpkin takes an admirable look at the hypocrisy of political correctness , but it does so with such an uneven tone that you never know when humor ends and tragedy begins . 
 True: Neg | Pred: Positive (79.8%) | the iditarod lasts for days - this just felt like it did . 
 True: Neg | Pred: Positive (85.5%) | holden caulfield did it better . 
 True: Neg | Pred: Positive (75.5%) | ( w ) hile long on amiable monkeys and worthy environmentalism , jane goodall 's wild chimpanzees is short on the thrills the oversize medium demands . 
 True: Pos | Pred: Negative (67.2%) | escaping the studio , piccoli is warmly affecting and so is this adroitly minimalist movie . 
 True: Neg | Pred: Positive (93.7%) | the title not only describes its m

In [16]:
#Task A
import os
import requests
import zipfile

#Завантаження GloVe embeddings
if not os.path.exists("glove.6B.100d.txt"):

    url = "https://nlp.stanford.edu/data/glove.6B.zip"

    print("Downloading GloVe embeddings...")

    r = requests.get(url)

    with open("glove.zip", "wb") as f:
        f.write(r.content)

    print("Extracting embeddings...")

    with zipfile.ZipFile("glove.zip", "r") as zip_ref:
        zip_ref.extractall(".")

def load_glove_embeddings(filepath, word2idx, emb_dim=100):
    """
    Функція завантажує GloVe embeddings
    та формує embedding матрицю відповідно до нашого словника.
    """

    #Створюємо випадкову матрицю embeddings
    embeddings = np.random.normal(scale=0.6, size=(len(word2idx), emb_dim))

    #PAD токен повинен бути нульовим
    embeddings[PAD_IDX] = np.zeros(emb_dim)

    found = 0

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:

            parts = line.strip().split()
            word = parts[0]

            #Якщо слово є в нашому словнику
            if word in word2idx:
                vector = np.array(parts[1:], dtype=np.float32)
                embeddings[word2idx[word]] = vector
                found += 1

    print(f"Знайдено embeddings для {found}/{len(word2idx)} слів")

    return torch.tensor(embeddings, dtype=torch.float32)


#Шлях до файлу glove
glove_path = "glove.6B.100d.txt"

#Завантаження embeddings
pretrained = load_glove_embeddings(glove_path, word2idx)

#Копіюємо embeddings у модель
model.embedding.weight.data.copy_(pretrained)

#Повторне навчання моделі
NUM_EPOCHS = 5

for epoch in range(1, NUM_EPOCHS + 1):

    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    val_loss, val_acc = evaluate(
        model, val_loader, criterion, device
    )

    print(f"Epoch {epoch} | Train Acc {train_acc:.2%} | Val Acc {val_acc:.2%}")


Extracting embeddings...
Знайдено embeddings для 13318/14311 слів
Epoch 1 | Train Acc 82.90% | Val Acc 81.77%
Epoch 2 | Train Acc 91.21% | Val Acc 82.45%
Epoch 3 | Train Acc 93.24% | Val Acc 83.14%
Epoch 4 | Train Acc 94.51% | Val Acc 84.86%
Epoch 5 | Train Acc 95.38% | Val Acc 83.72%


In [17]:
# Task B 

#Cтворюємо новий клас моделі з attention
class BiLSTMAttention(nn.Module):

    #Конструктор моделі
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_layers,
                 num_classes=2, dropout=0.3, pad_idx=PAD_IDX):

        super().__init__()  #Викликаємо конструктор базового класу nn.Module

        #embedding шар перетворює індекси слів у векторні представлення
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)

        #LSTM шар для обробки послідовності слів
        self.lstm = nn.LSTM(
            emb_dim,                     #розмір embedding вектора
            hidden_dim,                  #розмір прихованого стану
            num_layers=num_layers,       #кількість шарів LSTM
            batch_first=True,            #формат батчу: [batch, seq, features]
            bidirectional=True,          #використовуємо двонапрямлений LSTM
            dropout=dropout if num_layers > 1 else 0  #dropout між шарами
        )

        #dropout використовується для регуляризації
        self.dropout = nn.Dropout(dropout)

        #attention шар обчислює важливість кожного слова у послідовності
        self.attention = nn.Linear(hidden_dim * 2, 1)

        #фінальний повнозв'язний шар для класифікації
        self.fc = nn.Linear(hidden_dim * 2, num_classes)


    #forward проходження через модель
    def forward(self, x):

        #Перетворюємо токени у embedding вектори
        embedded = self.dropout(self.embedding(x))

        #Пропускаємо embeddings через BiLSTM
        outputs, _ = self.lstm(embedded)

        #outputs має форму [batch_size, seq_len, hidden_dim * 2]

        #Обчислюємо attention score для кожного токена
        attn_scores = self.attention(outputs)

        #Нормалізуємо scores у ймовірності
        attn_weights = torch.softmax(attn_scores, dim=1)

        #Обчислюємо контекстний вектор як зважену суму hidden states
        context = torch.sum(attn_weights * outputs, dim=1)

        #Застосовуємо dropout та класифікаційний шар
        logits = self.fc(self.dropout(context))

        #Повертаємо logits для подальшого обчислення втрат
        return logits

In [ ]:
# Task C 
# Cтворюємо клс моделі LSTM
class LSTMSentiment(nn.Module):

    #Конструктор моделі
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_layers,
                 num_classes=2, dropout=0.3, pad_idx=PAD_IDX):

        super().__init__() 

        #embedding шар перетворює індекси слів у векторні представлення
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)

        #Cтворюємо звичайний LSTM (НЕ bidirectional)
        self.lstm = nn.LSTM(
            emb_dim,                     
            hidden_dim,                  
            num_layers=num_layers,       
            batch_first=True,             
            bidirectional=False           
        )

        #dropout для регуляризації
        self.dropout = nn.Dropout(dropout)

        #Фінальний шар класифікації
        self.fc = nn.Linear(hidden_dim, num_classes)


    # forward проходження через модель
    def forward(self, x):

        #Перетворюємо індекси слів у embedding вектори
        embedded = self.embedding(x)

        #Передаємо embeddings у LSTM
        outputs, (hidden, _) = self.lstm(embedded)

        #hidden має форму [num_layers, batch_size, hidden_dim]

        #Беремо останній hidden state
        h_last = hidden[-1]

        #Застосовуємо dropout та класифікаційний шар
        logits = self.fc(self.dropout(h_last))

        #Повертаємо logits
        return logits

In [19]:
# Task D 

results = []  # список для збереження результатів експериментів


#Можливі значення embedding dimension
emb_dims = [50, 100, 200]

#Можливі значення hidden dimension
hidden_dims = [64, 128, 256]

#Можливі значення кількості LSTM шарів
num_layers_list = [1, 2]


#Перебираємо всі комбінації параметрів
for emb in emb_dims:

    for hidden in hidden_dims:

        for layers in num_layers_list:

            print(f"\nНавчання emb={emb}, hidden={hidden}, layers={layers}")

            #Створюємо модель з поточними гіперпараметрами
            model = BiLSTMSentiment(
                vocab_size=V,        #розмір словника
                emb_dim=emb,         #розмір embedding
                hidden_dim=hidden,   #розмір hidden state
                num_layers=layers    #кількість LSTM шарів
            ).to(device)             #переносимо модель на CPU або GPU


            #Створюємо оптимізатор Adam
            optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


            #Виконуємо одне навчання моделі
            train_loss, train_acc = train_one_epoch(
                model, train_loader, optimizer, criterion, device
            )


            #Оцінюємо модель на validation наборі
            val_loss, val_acc = evaluate(
                model, val_loader, criterion, device
            )


            #Зберігаємо результат експерименту
            results.append((emb, hidden, layers, val_acc))


#Виводимо таблицю результатів
print("\nТаблиця результатів експериментів")

#Проходимо по всіх результатах
for r in results:

    #r містить (emb_dim, hidden_dim, num_layers, accuracy)
    print(r)


Навчання emb=50, hidden=64, layers=1

Навчання emb=50, hidden=64, layers=2

Навчання emb=50, hidden=128, layers=1

Навчання emb=50, hidden=128, layers=2

Навчання emb=50, hidden=256, layers=1

Навчання emb=50, hidden=256, layers=2

Навчання emb=100, hidden=64, layers=1

Навчання emb=100, hidden=64, layers=2

Навчання emb=100, hidden=128, layers=1

Навчання emb=100, hidden=128, layers=2

Навчання emb=100, hidden=256, layers=1

Навчання emb=100, hidden=256, layers=2

Навчання emb=200, hidden=64, layers=1

Навчання emb=200, hidden=64, layers=2

Навчання emb=200, hidden=128, layers=1

Навчання emb=200, hidden=128, layers=2

Навчання emb=200, hidden=256, layers=1

Навчання emb=200, hidden=256, layers=2

Таблиця результатів експериментів
(50, 64, 1, 0.7442660550458715)
(50, 64, 2, 0.7591743119266054)
(50, 128, 1, 0.7580275229357798)
(50, 128, 2, 0.7786697247706422)
(50, 256, 1, 0.7706422018348624)
(50, 256, 2, 0.7178899082568807)
(100, 64, 1, 0.7775229357798165)
(100, 64, 2, 0.7763761467889